## encodings: PANDA x UNI2-h / Virchow2

PANDA ships as raw gigapixel WSIs, not pre-cut patches, so this notebook does three things the other two don't need:

1. **Tile** every slide into an exhaustive grid of foreground tissue patches (Otsu foreground detection, prototyped in `exploration/explore_panda.ipynb`) -- every non-overlapping `patch_size x patch_size` tile whose Otsu foreground coverage clears a threshold, not a random sample.
2. **Embed** those patches with UNI2-h and Virchow2, streaming through each slide in bounded-memory chunks so neither slide count nor patches-per-slide is capped.
3. **Resume safely**: each slide's patch encodings are written to their own file the moment they're done (atomically, so a crash mid-write can't leave a corrupt file that looks finished), and any slide whose output file already exists is skipped on the next run -- an interrupted run (crash, timeout, manual stop) can just be restarted.

Progress is logged to `encodings/encode_panda.log` as well as printed in the notebook, so the run can be monitored from another terminal with `tail -f` while it works through the full dataset -- for the full PANDA release (~10k slides), this is a long-running job, likely hours on a single GPU.

Caveat: patches are extracted at level-0 (native resolution) without checking each slide's microns-per-pixel, so magnification isn't calibrated across slides/scanners the way a production pipeline would want -- fine for a first pass, worth revisiting if results look off between Radboud and Karolinska slides.

In [ ]:
!du -sh /home/shared/data/panda

In [ ]:
import os

os.environ["HF_HOME"] = "/home/shared/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/home/shared/.cache/huggingface/hub"
os.environ["TRANSFORMERS_CACHE"] = (
    "/home/shared/.cache/huggingface/hub"  # deprecated alias, harmless to set
)

In [ ]:
!pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys
import pyrootutils

# VS Code's Jupyter kernel starts in the launch dir, not the notebook's folder,
# so an upward search from cwd can miss the repo root -- search from the notebook
# file itself (VS Code sets __vsc_ipynb_file__) and fall back to cwd otherwise.
search_from = globals().get("__vsc_ipynb_file__", ".")
root = pyrootutils.setup_root(search_from=search_from, indicator=".project-root", pythonpath=False)
sys.path.append(str(root / "src"))
from vfm_encoders import load_uni2, load_virchow2, embed_images

import os
import time
import logging
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import openslide
from pathlib import Path
from tqdm.auto import tqdm

### 0. Setup

Both encoders are gated on Hugging Face -- request access on the model pages, then set `HF_TOKEN` (or leave unset for an interactive login prompt).

In [ ]:
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. Load slide-level metadata

Prefers `train_subset.csv` (the stratified subset from the download notebook) if it exists on disk, otherwise falls back to the full `train.csv` filtered to slides actually present locally. Every slide found here gets processed -- there's no further sampling downstream.

In [ ]:
panda_base_dir = Path("/home/shared/data/panda/")
train_images_dir = panda_base_dir / "train_images"

csv_path = panda_base_dir / "train_subset.csv"
if not csv_path.exists():
    csv_path = panda_base_dir / "train.csv"

train_df = (
    pd.read_csv(csv_path)
    .assign(filepath=lambda x: x["image_id"].apply(lambda y: str(train_images_dir / f"{y}.tiff")))
    .assign(exists=lambda x: x["filepath"].apply(lambda y: Path(y).exists()))
    .loc[lambda x: x["exists"]]
    .sort_values("image_id")
    .reset_index(drop=True)
)
print(f"Slides available: {len(train_df)}")
train_df["isup_grade"].value_counts().sort_index()

### 2. Configure logging

Every slide's outcome (encoded / skipped / failed) is logged to a file, not just printed -- so progress can be checked from another terminal (or after the notebook's own output has scrolled away) with:

```bash
tail -f /home/shared/data/panda/encodings/encode_panda.log
```

In [ ]:
out_dir = panda_base_dir / "encodings"
out_dir.mkdir(parents=True, exist_ok=True)
log_path = out_dir / "encode_panda.log"

logger = logging.getLogger("encode_panda")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate lines if this cell gets re-run

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")

file_handler = logging.FileHandler(log_path)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

print(f"Logging to {log_path}")
print(f"Watch progress from another terminal with: tail -f {log_path}")

### 3. Load the encoders

In [ ]:
encoders = {
    "uni2-h": load_uni2(device),
    "virchow2": load_virchow2(device),
}
for enc in encoders.values():
    n_params = sum(p.numel() for p in enc.model.parameters())
    print(f"{enc.name:10s} embed_dim={enc.embed_dim:5d}  params={n_params / 1e6:.0f}M")

### 4. Tiling: grid-tile every foreground patch

Same Otsu foreground mask as `explore_panda.ipynb`, but instead of sampling random points in the foreground, this walks a non-overlapping `patch_size x patch_size` grid over the *entire* slide and keeps every tile whose foreground coverage clears `foreground_threshold` -- exhaustive, so every patch of tissue on the slide gets encoded exactly once (not zero, not twice).

In [ ]:
def tile_foreground_patches(
    slide: openslide.OpenSlide,
    patch_size: int = 224,
    foreground_threshold: float = 0.5,
    thumbnail_size: int = 1024,
) -> list[tuple[int, int]]:
    """Returns level-0 (x0, y0) top-left coordinates for every non-overlapping
    patch_size x patch_size tile whose Otsu foreground coverage is at least
    `foreground_threshold`."""
    width, height = slide.dimensions
    thumbnail = slide.get_thumbnail((thumbnail_size, thumbnail_size))
    thumb_arr = np.array(thumbnail.convert("L"))

    _, foreground_mask = cv2.threshold(thumb_arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    foreground_mask = foreground_mask > 0

    scale_x = width / thumb_arr.shape[1]
    scale_y = height / thumb_arr.shape[0]

    coords = []
    for y0 in range(0, height - patch_size + 1, patch_size):
        my0 = int(y0 / scale_y)
        my1 = max(my0 + 1, int((y0 + patch_size) / scale_y))
        for x0 in range(0, width - patch_size + 1, patch_size):
            mx0 = int(x0 / scale_x)
            mx1 = max(mx0 + 1, int((x0 + patch_size) / scale_x))
            if foreground_mask[my0:my1, mx0:mx1].mean() >= foreground_threshold:
                coords.append((x0, y0))
    return coords

### 5. Chunked per-slide encoding

A single slide can have thousands of foreground tiles, so patches are read from `openslide` and embedded chunk by chunk rather than all materialized as PIL images up front -- same reasoning as `encode_pathmnist.ipynb`'s `encode_split_in_chunks`, just per-slide instead of per-dataset-split.

In [ ]:
def encode_slide_patches(
    slide: openslide.OpenSlide,
    coords: list[tuple[int, int]],
    encoders: dict,
    device: str,
    patch_size: int = 224,
    chunk_size: int = 256,
) -> dict[str, np.ndarray]:
    """Returns {encoder_name: (len(coords), embed_dim) array}."""
    n = len(coords)
    out = {enc.name: np.empty((n, enc.embed_dim), dtype=np.float32) for enc in encoders.values()}

    for start in range(0, n, chunk_size):
        chunk_coords = coords[start : start + chunk_size]
        patches = [
            slide.read_region((x0, y0), level=0, size=(patch_size, patch_size)).convert("RGB")
            for x0, y0 in chunk_coords
        ]
        for enc in encoders.values():
            out[enc.name][start : start + len(patches)] = embed_images(
                enc, patches, device=device, show_progress=False
            )

    return out

### 6. Encode every slide, skipping ones already done

For each slide: skip immediately if `patches/{image_id}.npz` already exists (from a previous run of this cell); otherwise tile, encode, and write it -- to a temp file first, then `os.replace` into place, so a run that gets killed mid-slide never leaves a half-written file that a later run would mistake for done.

`DEBUG_MAX_SLIDES` is the only cap left -- set it to a small number for a quick smoke test; leave it `None` to process every slide in `train_df`.

In [ ]:
PATCH_SIZE = 224
FOREGROUND_THRESHOLD = 0.5
DEBUG_MAX_SLIDES = None  # e.g. 5 for a smoke test; None processes every slide

slides_to_process = train_df if DEBUG_MAX_SLIDES is None else train_df.head(DEBUG_MAX_SLIDES)
patches_dir = out_dir / "patches"
patches_dir.mkdir(parents=True, exist_ok=True)

n_encoded, n_skipped, n_failed = 0, 0, 0
total_new_patches = 0
failed_slides = []
run_start = time.time()

logger.info(
    f"Starting run: {len(slides_to_process)} slides, patch_size={PATCH_SIZE}, "
    f"foreground_threshold={FOREGROUND_THRESHOLD}"
)

for i, row in enumerate(
    tqdm(slides_to_process.itertuples(), total=len(slides_to_process), desc="encoding slides")
):
    out_path = patches_dir / f"{row.image_id}.npz"
    progress = f"[{i + 1}/{len(slides_to_process)}]"

    if out_path.exists():
        n_skipped += 1
        logger.info(f"{progress} {row.image_id}: already encoded, skipping")
        continue

    slide_start = time.time()
    try:
        slide = openslide.OpenSlide(row.filepath)
        try:
            coords = tile_foreground_patches(
                slide, patch_size=PATCH_SIZE, foreground_threshold=FOREGROUND_THRESHOLD
            )
            if not coords:
                logger.warning(f"{progress} {row.image_id}: no foreground patches found, skipping")
                n_failed += 1
                failed_slides.append(row.image_id)
                continue

            feats = encode_slide_patches(slide, coords, encoders, device, patch_size=PATCH_SIZE)
        finally:
            slide.close()

        x0s, y0s = zip(*coords)
        tmp_path = out_path.with_suffix(".npz.tmp")
        np.savez(tmp_path, x0=np.array(x0s), y0=np.array(y0s), **feats)
        os.replace(tmp_path, out_path)

        n_encoded += 1
        total_new_patches += len(coords)
        logger.info(
            f"{progress} {row.image_id}: {len(coords)} foreground patches encoded in "
            f"{time.time() - slide_start:.1f}s"
        )
    except Exception:
        n_failed += 1
        failed_slides.append(row.image_id)
        logger.exception(f"{progress} {row.image_id}: failed, skipping")

logger.info(
    f"Run done in {(time.time() - run_start) / 60:.1f} min: {n_encoded} encoded, "
    f"{n_skipped} already done, {n_failed} failed, {total_new_patches:,} new patches"
)
if failed_slides:
    logger.warning(f"Failed/empty slides: {failed_slides}")

### 7. Combine per-slide outputs into patch-level + slide-level files

Reads whatever `patches/*.npz` files exist right now -- every slide encoded so far, whether from this run or an earlier interrupted one -- and writes the same patch-level `.npy` + `.csv` and slide-level (mean-pooled) `.csv` outputs the other `02_encodings` notebooks produce. Safe to re-run at any time (including while the cell above is still running in another session) to get an up-to-date snapshot without waiting for the full dataset to finish.

In [ ]:
def combine_patch_encodings(patches_dir: Path, train_df: pd.DataFrame, model_names: list[str]):
    npz_paths = sorted(patches_dir.glob("*.npz"))
    if not npz_paths:
        raise FileNotFoundError(
            f"No per-slide encodings found in {patches_dir} yet -- run the encoding cell first."
        )

    slide_meta = train_df.set_index("image_id")[["isup_grade", "data_provider"]]

    feats = {name: [] for name in model_names}
    meta_rows = []
    for p in npz_paths:
        image_id = p.stem
        data = np.load(p)
        for name in model_names:
            feats[name].append(data[name])
        row = slide_meta.loc[image_id]
        meta_rows.append(
            pd.DataFrame(
                {
                    "image_id": image_id,
                    "isup_grade": row["isup_grade"],
                    "data_provider": row["data_provider"],
                    "x0": data["x0"],
                    "y0": data["y0"],
                }
            )
        )

    patch_metadata = pd.concat(meta_rows, ignore_index=True)
    combined_feats = {name: np.concatenate(arrs, axis=0) for name, arrs in feats.items()}
    return combined_feats, patch_metadata


MODEL_NAMES = ["uni2-h", "virchow2"]
combined_feats, patch_metadata = combine_patch_encodings(patches_dir, train_df, MODEL_NAMES)

n_done = len(list(patches_dir.glob("*.npz")))
logger.info(
    f"Combined {len(patch_metadata):,} patches from {patch_metadata['image_id'].nunique():,} "
    f"slides ({n_done} of {len(train_df)} slides encoded so far)"
)

patch_metadata.to_csv(out_dir / "patch_metadata_all.csv", index=False)

slide_labels = train_df.set_index("image_id")[["isup_grade", "data_provider"]]
for model_name, feats_arr in combined_feats.items():
    np.save(out_dir / f"{model_name}_patch_all.npy", feats_arr)

    slide_ids = patch_metadata["image_id"].values
    slide_feats = pd.DataFrame(feats_arr, index=slide_ids).groupby(level=0).mean()
    slide_feats = slide_feats.join(slide_labels, how="left")
    slide_feats.to_csv(out_dir / f"{model_name}_slide_all.csv")

    print(f"Saved {model_name}: {feats_arr.shape[0]:,} patches, {len(slide_feats):,} slides")

### 8. Sanity check

PCA of the slide-level (mean-pooled) encodings, colored by ISUP grade -- higher grades should drift away from grade 0 if the encoder is picking up tumor morphology.

In [ ]:
slide_feats

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

fig, axs = plt.subplots(1, len(MODEL_NAMES), figsize=(6 * len(MODEL_NAMES), 5))
for ax, model_name in zip(axs, MODEL_NAMES):
    slide_feats = pd.read_csv(out_dir / f"{model_name}_slide_all.csv", index_col=0)
    feat_cols = [c for c in slide_feats.columns if c not in ("isup_grade", "data_provider")]
    coords = PCA(n_components=2, random_state=42).fit_transform(slide_feats[feat_cols])

    sc = sns.scatterplot(
        x=coords[:, 0],
        y=coords[:, 1],
        hue=slide_feats["isup_grade"],
        palette="RdYlGn_r",
        style=slide_feats["data_provider"],
        s=100,
        legend=ax == axs[-1],
        ax=ax,
    )
    ax.set_title(f"{model_name} -- slide-level, by ISUP grade")

sns.move_legend(ax, bbox_to_anchor=(1, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

fig, axs = plt.subplots(1, len(MODEL_NAMES), figsize=(6 * len(MODEL_NAMES), 5))
for ax, model_name in zip(axs, MODEL_NAMES):
    slide_feats = pd.read_csv(out_dir / f"{model_name}_slide_all.csv", index_col=0)
    feat_cols = [c for c in slide_feats.columns if c not in ("isup_grade", "data_provider")]
    coords = PCA(n_components=2, random_state=42).fit_transform(slide_feats[feat_cols])

    sc = sns.scatterplot(
        x=coords[:, 0],
        y=coords[:, 1],
        hue=slide_feats["data_provider"],
        palette="tab10",
        s=40,
        ax=ax,
    )
    ax.set_title(f"{model_name} -- slide-level, by data provider")

plt.tight_layout()
plt.show()